# Проверка API сомелье

Запустите сервис через `docker compose up --build -d` из корня проекта. Затем выполняйте ячейки по порядку. Ключ OpenRouter в notebook не нужен.

In [ ]:
import os
import secrets
import time
import uuid

import httpx
from IPython.display import JSON, display

BASE_URL = os.environ.get("SOMMELIER_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
client = httpx.Client(base_url=BASE_URL, timeout=90.0, trust_env=False)

def show(response: httpx.Response):
    print(f"{response.request.method} {response.request.url} → {response.status_code}")
    if response.content:
        payload = response.json()
        display(JSON(payload))
    else:
        payload = None
    response.raise_for_status()
    return payload


In [ ]:
show(client.get("/health"))

## Новый диалог

Измените `wine` и `candidates` на реальные карточки каталога. Каждое выполнение следующей ячейки создаёт новый UUIDv7 и новую сессию.

In [ ]:
def new_uuid7() -> uuid.UUID:
    # Python 3.13 ещё не предоставляет uuid.uuid7().
    timestamp_ms = time.time_ns() // 1_000_000
    random_bits = secrets.randbits(74)
    value = (timestamp_ms & ((1 << 48) - 1)) << 80
    value |= 7 << 76
    value |= ((random_bits >> 62) & 0xFFF) << 64
    value |= 0b10 << 62
    value |= random_bits & ((1 << 62) - 1)
    return uuid.UUID(int=value)

session_id = str(new_uuid7())
wine = {
    "Название вина": "Кокур",
    "Категория": "Белое",
    "Регион": "Крым",
    "Сорт винограда": "Кокур",
    "Slug": "kokur",
}
candidates = [
    {"Название вина": "Другой кокур", "Slug": "other-kokur"},
]
opening = show(client.post(
    f"/v1/sessions/{session_id}",
    json={"wine": wine, "candidates": candidates},
))
print("session_id:", session_id)
print("Подсказки:", opening["message"]["suggestions"])

In [ ]:
question = "С чем лучше сочетать это вино?"
reply = show(client.post(
    f"/v1/sessions/{session_id}/messages",
    json={"content": question},
))

In [ ]:
history = show(client.get(f"/v1/sessions/{session_id}"))

## Необязательно: удалить сессию

Запустите следующую ячейку, когда история больше не нужна.

In [ ]:
show(client.delete(f"/v1/sessions/{session_id}"))